In [ ]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
import os
import zipfile
from datetime import datetime
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# Função para limpar caracteres ilegais do Excel
def remover_caracteres_invalidos(texto):
    if not isinstance(texto, str):
        return texto
    return ILLEGAL_CHARACTERS_RE.sub("", texto)

# Categorias da Copafer
categorias = [
    "banheiro-e-cozinha", "especial", "ferragens", "ferramentas", "iluminacao",
    "jardim-e-lazer", "materiais-eletricos", "material-hidraulico", "materiais-para-construcao",
    "material-automotivo", "pet-shop", "pisos-e-revestimentos", "portas-e-janelas",
    "tintas-e-acessorios", "utilidades-domesticas"
]

# Configurações gerais
headers = {"User-Agent": "Mozilla/5.0"}
limite_api = 50  # Lote por página
data_hoje = datetime.now().strftime("%Y-%m-%d")
pasta_base = f"saida_copafer_{data_hoje}"
os.makedirs(pasta_base, exist_ok=True)

# Início da coleta por categoria
for categoria in categorias:
    print(f"\n🔍 Coletando categoria: {categoria}")
    todos_produtos = []
    pagina = 0

    while True:
        url = f"https://www.copafer.com.br/api/catalog_system/pub/products/search/{categoria}?O=OrderByReleaseDateDESC&_from={pagina * limite_api}&_to={(pagina + 1) * limite_api - 1}"
        print(f"   📄 Página {pagina + 1}...")

        resposta = requests.get(url, headers=headers)
        if resposta.status_code not in [200, 206]:
            print(f"   ❌ Erro: {resposta.status_code}")
            break

        produtos = resposta.json()
        if not produtos:
            print("   ✅ Fim da categoria.")
            break

        for produto in produtos:
            try:
                nome = produto.get("productName", "").strip()
                raw_link = produto.get("link", "")
                link = raw_link if raw_link.startswith("http") else "https://www.copafer.com.br" + raw_link
                codigo = produto.get("productReference", "")
                preco_promocional = produto["items"][0]["sellers"][0]["commertialOffer"]["Price"]
                preco_original = produto["items"][0]["sellers"][0]["commertialOffer"]["ListPrice"]
                preco_m2 = round(preco_promocional / produto["items"][0].get("unitMultiplier", 1), 2)

                descricao_html = produto.get("description", "")
                descricao_limpa = BeautifulSoup(descricao_html, "html.parser").get_text(separator=" ").strip()
                descricao_limpa = ' '.join(descricao_limpa.split())

                marca = produto.get("brand", "").strip()
                especificacoes = f"Marca: {marca}"

                imagem = produto["items"][0]["images"][0]["imageUrl"]
                loja = "Copafer"

                todos_produtos.append({
                    "Código": codigo,
                    "Nome": nome,
                    "Marca": marca,
                    "Preço Promocional": preco_promocional,
                    "Preço Original": preco_original,
                    "Preço por m²": preco_m2,
                    "Especificações": especificacoes,
                    "Descrição": descricao_limpa,
                    "Imagem": imagem,
                    "URL": link,
                    "Categoria": categoria,
                    "Loja": loja
                })

            except Exception as e:
                print(f"   ⚠️ Erro ao processar produto: {e}")

        pagina += 1
        time.sleep(0.5)

    # Criar pasta por categoria
    pasta_categoria = os.path.join(pasta_base, categoria.replace("/", "_"))
    os.makedirs(pasta_categoria, exist_ok=True)

    # Salvar CSV e Excel
    csv_path = os.path.join(pasta_categoria, f"{categoria}_copafer.csv")
    xlsx_path = os.path.join(pasta_categoria, f"{categoria}_copafer.xlsx")
    df = pd.DataFrame(todos_produtos)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    # Corrigir caracteres ilegais antes de salvar Excel
    df_limpo = df.applymap(remover_caracteres_invalidos)
    df_limpo.to_excel(xlsx_path, index=False)

    # Baixar imagens
    images_dir = os.path.join(pasta_categoria, "imagens")
    os.makedirs(images_dir, exist_ok=True)

    for _, row in df.iterrows():
        codigo = row["Código"]
        img_url = row["Imagem"]
        if img_url != "N/A":
            tentativas = 0
            sucesso = False
            while tentativas < 3 and not sucesso:
                try:
                    resp = requests.get(img_url, stream=True, timeout=10)
                    if resp.status_code == 200:
                        imagem_path = os.path.join(images_dir, f"{codigo}.png")
                        with open(imagem_path, "wb") as f:
                            for chunk in resp.iter_content(1024):
                                f.write(chunk)
                        sucesso = True
                    else:
                        print(f"❌ Erro ao baixar imagem de {codigo}")
                except Exception as e:
                    print(f"❌ Exceção ao baixar imagem: {e}")
                finally:
                    if not sucesso:
                        tentativas += 1
                        time.sleep(5)

    # Compactar imagens
    zip_filename = os.path.join(pasta_categoria, f"imagens_{categoria}.zip")
    with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
        for filename in os.listdir(images_dir):
            zipf.write(os.path.join(images_dir, filename), filename)

    print(f"✅ Categoria '{categoria}' finalizada com {len(todos_produtos)} produtos.")
    print(f"   📄 CSV salvo: {csv_path}")
    print(f"   📊 Excel salvo: {xlsx_path}")
    print(f"   📦 Imagens ZIP: {zip_filename}")
